# Training Model Klasifikasi Citra Sampah (CNN MobileNetV2)

Dibuat oleh:

- Nama     : Zaiz Febrian
- NIM      : 10122052
- Prodi    : Teknik Informatika
- Fakultas : Fakultas Teknik dan Ilmu Komputer
- Kampus   : Universitas Komputer Indonesia

Studi Kasus : UPT TPSA Cihara
Metode      : CNN MobileNetV2 (transfer learning) dengan Data Augmentation

## Gambaran alur (baca dulu)

Urutan langkah dalam notebook ini:

1. Siapkan foto sampah, satu folder untuk satu jenis (kelas).
2. Kalau foto masih HEIC dari iPhone, ubah dulu jadi jpg.
3. Pilah atau potong tiap objek lalu simpan ke folder kelasnya.
4. Naikkan dataset ke Colab (upload zip atau mount Google Drive).
5. Bagi data jadi train, validation, dan test.
6. Latih model.
7. Lihat grafik dan evaluasi hasilnya.
8. Simpan model lalu coba jalankan untuk prediksi.

Kelas awal: organik, plastik, kertas, logam, kaca.
Jumlah kelas bebas, sistem membaca otomatis dari nama folder.

## 1. Cek GPU dan import library

GPU kalo bisa aktif supaya training lebih cepat.

In [ ]:
import os
import json
import shutil
import random
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Input, layers, Model, Sequential

print("Versi TensorFlow:", tf.__version__)
gpu = tf.config.list_physical_devices("GPU")
print("GPU terdeteksi:", gpu if gpu else "tidak ada (jalan di CPU, lebih lambat)")

## 2. Pengaturan dasar

Ubah nilai di bawah ini kalo perlu

In [ ]:
# Ukuran gambar yang dipakai MobileNetV2
IMG_SIZE = (224, 224)
CHANNELS = 3

# Ukuran batch dan jumlah putaran latihan
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 1e-4

# Angka acak tetap supaya hasil bisa diulang
SEED = 42
random.seed(SEED)
tf.random.set_seed(SEED)

# Lokasi folder dataset di dalam Colab
BASE_DIR = pathlib.Path("dataset")
RAW_DIR = BASE_DIR / "raw"            # foto mentah, satu folder per kelas
TRAIN_DIR = BASE_DIR / "train"
VAL_DIR = BASE_DIR / "validation"
TEST_DIR = BASE_DIR / "test"

print("Folder dataset:", BASE_DIR.resolve())

## 3. Siapkan dataset

Susun foto seperti ini, satu folder untuk satu jenis sampah:

    dataset/raw/organik/   foto1.jpg, foto2.jpg, ...
    dataset/raw/plastik/   ...
    dataset/raw/kertas/    ...
    dataset/raw/logam/     ...
    dataset/raw/kaca/      ...

Format yang didukung: jpg, jpeg, png. Saran jumlah minimal sekitar 50 gambar
per kelas, lebih banyak lebih baik, dan usahakan jumlahnya mirip antar kelas.

Catatan satu foto banyak objek: foto TPSA sering berisi banyak sampah dalam
satu frame. Untuk klasifikasi (satu gambar satu kelas) bagusnya dipotong tiap objek, terus simpan ke folder kelasnya.

Pilih salah satu cara upload dataset ke Colab pada sel berikut.

In [ ]:
# Cara A: upload file dataset.zip dari komputer (struktur di dalamnya: raw/<kelas>/...)
# Hapus tanda pagar pada blok di bawah bila ingin memakai cara ini.

# from google.colab import files
# uploaded = files.upload()            # pilih dataset.zip
# shutil.unpack_archive("dataset.zip", str(BASE_DIR))


# Cara B: ambil dari Google Drive.
# Hapus tanda pagar lalu sesuaikan path sumbernya.

# from google.colab import drive
# drive.mount("/content/drive")
# sumber = "/content/drive/MyDrive/dataset_sampah/raw"   # ganti sesuai punya Anda
# shutil.copytree(sumber, str(RAW_DIR), dirs_exist_ok=True)


# Cek hasilnya
if RAW_DIR.exists():
    kelas = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
    print("Kelas ditemukan:", kelas)
    for k in kelas:
        jumlah = len(list((RAW_DIR / k).glob("*")))
        print(f"  {k}: {jumlah} gambar")
else:
    print("Folder", RAW_DIR, "belum ada. Naikkan dataset dulu memakai cara A atau B.")

## 4. Bagi data jadi train, validation, dan test

Pembagian dibuat per kelas supaya tiap jenis tetap seimbang.
Rasio default 70 persen latih, 15 persen validasi, 15 persen uji.

In [ ]:
RATIOS = (0.70, 0.15, 0.15)
EXTS = {".jpg", ".jpeg", ".png"}

def bagi_dataset(raw_dir, train_dir, val_dir, test_dir, ratios, seed):
    # Kosongkan hasil split lama supaya tidak tercampur
    for d in (train_dir, val_dir, test_dir):
        if d.exists():
            shutil.rmtree(d)

    rng = random.Random(seed)
    kelas = sorted([d.name for d in raw_dir.iterdir() if d.is_dir()])

    for nama in kelas:
        berkas = [p for p in (raw_dir / nama).iterdir()
                  if p.suffix.lower() in EXTS]
        rng.shuffle(berkas)

        n = len(berkas)
        n_train = int(n * ratios[0])
        n_val = int(n * ratios[1])

        bagian = {
            train_dir: berkas[:n_train],
            val_dir: berkas[n_train:n_train + n_val],
            test_dir: berkas[n_train + n_val:],
        }
        for tujuan, daftar in bagian.items():
            (tujuan / nama).mkdir(parents=True, exist_ok=True)
            for sumber in daftar:
                shutil.copy2(sumber, tujuan / nama / sumber.name)

        print(f"{nama}: total {n}, train {n_train}, val {n_val}, "
              f"test {n - n_train - n_val}")

bagi_dataset(RAW_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR, RATIOS, SEED)
print("Selesai membagi data.")

## 5. Muat dataset jadi batch

Gambar dibaca dari folder dan diubah jadi kumpulan batch siap latih.
Nama kelas diambil otomatis dari nama folder.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, label_mode="int")

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    shuffle=False, label_mode="int")

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    shuffle=False, label_mode="int")

class_names = train_ds.class_names
num_classes = len(class_names)
print("Daftar kelas:", class_names)

# Prefetch supaya pembacaan data tidak menghambat GPU
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 6. Lihat contoh gambar

Sekadar memastikan gambar dan labelnya terbaca dengan benar.

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Data augmentation

Augmentasi membuat variasi gambar (balik, putar, zoom, geser, kontras)
hanya saat latihan. Tujuannya supaya model tidak gampang hafal dan lebih tahan
terhadap foto baru. Saat prediksi, augmentasi otomatis mati.

In [ ]:
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
], name="data_augmentation")

## 8. Bangun model MobileNetV2

Memakai transfer learning: MobileNetV2 yang sudah terlatih di ImageNet
dipakai sebagai dasar (dibekukan), lalu ditambah lapisan klasifikasi sendiri.

Layer Rescaling mengubah piksel dari rentang 0 sampai 255 menjadi -1 sampai 1.
Normalisasi ini sengaja ditaruh di dalam model supaya saat prediksi cukup
memberi gambar mentah, tanpa langkah tambahan yang gampang keliru.

In [ ]:
inputs = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], CHANNELS))

# Augmentasi (aktif saat latihan saja)
x = data_augmentation(inputs)

# Normalisasi MobileNetV2: 0..255 menjadi -1..1
x = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0,
                     name="mobilenetv2_preprocess")(x)

# Model dasar pretrained, dibekukan
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], CHANNELS),
    include_top=False, weights="imagenet")
base_model.trainable = False
x = base_model(x, training=False)

# Kepala klasifikasi
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs, name="waste_classifier")
model.summary()

## 9. Latih model

Model latihan disimpan otomatis ke waste_classifier.keras setiap kali
akurasi validasi membaik. Latihan berhenti lebih awal bila tidak ada
perbaikan, supaya hemat waktu.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "waste_classifier.keras", monitor="val_accuracy",
        save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
]

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks)

## 10. Grafik akurasi dan loss

Garis latih dan validasi yang berdekatan menandakan model belajar dengan sehat.

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epoch_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_range, acc, label="Latih")
plt.plot(epoch_range, val_acc, label="Validasi")
plt.title("Akurasi")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_range, loss, label="Latih")
plt.plot(epoch_range, val_loss, label="Validasi")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 11. Evaluasi pakai data test

Data test belum pernah dilihat model, jadi angkanya menggambarkan
kemampuan sebenarnya. Confusion matrix menunjukkan kelas mana yang
sering tertukar.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

loss_test, acc_test = model.evaluate(test_ds)
print(f"Akurasi pada data test: {acc_test * 100:.2f} persen")

y_true = []
y_pred = []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(labels.numpy())

print()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_xlabel("Prediksi")
ax.set_ylabel("Sebenarnya")
ax.set_title("Confusion Matrix")

for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im)
plt.tight_layout()
plt.show()

## 12. Simpan model dan label

File yang dihasilkan:

- waste_classifier.keras : model hasil latihan
- labels.json            : pemetaan nomor ke nama kelas

Kedua file inilah yang nanti dipakai aplikasi web.

In [ ]:
# Pastikan menyimpan model terbaik (bobot terbaik sudah dikembalikan EarlyStopping)
model.save("waste_classifier.keras")

labels = {str(i): nama for i, nama in enumerate(class_names)}
with open("labels.json", "w") as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)

print("Tersimpan: waste_classifier.keras dan labels.json")
print("Label:", labels)

## 13. Coba jalankan model (prediksi satu gambar)

Memuat ulang model dari file lalu menebak satu gambar, untuk memastikan
model siap dipakai.

In [ ]:
model_uji = tf.keras.models.load_model("waste_classifier.keras")

with open("labels.json") as f:
    labels = json.load(f)

def prediksi(path_gambar):
    img = tf.keras.utils.load_img(path_gambar, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)          # bentuk (1, 224, 224, 3)

    probs = model_uji.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))

    print("Gambar    :", path_gambar)
    print("Prediksi  :", labels[str(idx)])
    print(f"Confidence: {probs[idx] * 100:.2f} persen")
    print("Semua kelas:")
    for i, p in enumerate(probs):
        print(f"  {labels[str(i)]:10s}: {p * 100:.2f} persen")

# Ambil satu gambar contoh dari folder test
contoh = next(TEST_DIR.rglob("*.jpg"), None)
if contoh:
    prediksi(contoh)
else:
    print("Tidak menemukan gambar contoh di folder test.")

# Untuk mencoba gambar sendiri di Colab, hapus tanda pagar di bawah:
# from google.colab import files
# naik = files.upload()
# for nama in naik:
#     prediksi(nama)

## 14. Pakai model di aplikasi web

1. Unduh dua file hasil notebook ini: waste_classifier.keras dan labels.json.
   Di Colab bisa pakai:

       from google.colab import files
       files.download("waste_classifier.keras")
       files.download("labels.json")

2. Letakkan keduanya di folder model pada project (model/waste_classifier.keras
   dan model/labels.json).

3. Jalankan aplikasi web dari terminal:

       python app/app.py

4. Buka http://127.0.0.1:5000 di browser, unggah foto, lalu lihat hasilnya.

Kalau model belum ada, aplikasi otomatis memakai mode demo (MobileNetV2 ImageNet)
hanya untuk mencoba tampilan.

## Penutup

Notebook training selesai. Untuk hasil yang lebih baik, tambah jumlah dan
variasi gambar tiap kelas, jaga jumlahnya seimbang, lalu jalankan ulang.

Disusun oleh Zaiz Febrian (10122052), Teknik Informatika,
Fakultas Teknik dan Ilmu Komputer, Universitas Komputer Indonesia.
Studi Kasus UPT TPSA Cihara.